## Project : Astra 


In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langsmith import traceable
from langgraph.graph import StateGraph, END
from typing import TypedDict
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "astra-rag-optimizer"

print("All imports successful")

All imports successful


In [2]:
## Load documents : 
loader = PyPDFLoader("1706.03762v7.pdf")
docs = loader.load()
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nluk

In [3]:
PROMPT_VARIANTS = {
    "v1": ChatPromptTemplate.from_messages([
        ("system", """Answer the following question based only on the provided context.
Think step by step before providing an answer.
<context>
{context}
</context>"""),
        ("human", "{input}")
    ]),

    "v2": ChatPromptTemplate.from_messages([
        ("system", """You are a precise research assistant. 
Use ONLY the context below. If the answer is not in the context, say 'Not found in context'.
Be concise and cite specific parts of the context.
<context>
{context}
</context>"""),
        ("human", "{input}")
    ]),

    "v3": ChatPromptTemplate.from_messages([
        ("system", """Answer the question using the context provided.
First identify the most relevant sentences in the context.
Then construct your answer from those sentences only.
<context>
{context}
</context>"""),
        ("human", "{input}")
    ])
}
print("Prompt variants ready: v1, v2, v3")



Prompt variants ready: v1, v2, v3


In [10]:
def build_rag_pipeline(docs, chunk_size: int, chunk_overlap: int, top_k: int, prompt_variant: str):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)

    db = FAISS.from_documents(chunks, OpenAIEmbeddings())
    retriever = db.as_retriever(search_kwargs={"k": top_k})

    prompt = PROMPT_VARIANTS[prompt_variant]
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

    document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
    retrieval_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

    print(f"Pipeline built → chunk_size={chunk_size}, top_k={top_k}, prompt={prompt_variant}, chunks={len(chunks)}")
    return retrieval_chain

In [11]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at reformulating search queries.
Given a user question, generate exactly 3 different search queries to retrieve relevant information.
Return ONLY the 3 queries, one per line, numbered 1. 2. 3. No extra explanation."""),
    ("human", "Original question: {question}")
])

rewriter_chain = (
    rewrite_prompt| ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)| StrOutputParser()
)

def rewrite_query(question: str) -> list:
    raw = rewriter_chain.invoke({"question": question})
    lines = [l.strip() for l in raw.strip().split("\n") if l.strip()]
    cleaned = []
    for line in lines:
        if line and line[0].isdigit():
            line = line[2:].strip() if len(line) > 2 and line[1] in ".)" else line
        cleaned.append(line)
    return cleaned[:3]

# Test
print(rewrite_query("What is self-attention?"))

['What is the concept of self-attention in neural networks?', 'How does self-attention mechanism work in deep learning models?', 'Explain the importance of self-attention in natural language processing tasks.']


In [13]:
# NEW CELL — runs one question through rewriter + retrieval chain
@traceable(name="run-rag-question")
def run_rag(question: str, chain) -> dict:
    sub_queries = rewrite_query(question)

    all_contexts = []
    seen = set()
    for q in sub_queries:
        result = chain.invoke({"input": q})
        for doc in result.get("context", []):
            content = doc.page_content
            if content not in seen:
                seen.add(content)
                all_contexts.append(content)

    final_result = chain.invoke({"input": question})

    return {
    "question": question,
    "answer": final_result["answer"],
    "contexts": all_contexts if all_contexts else [final_result["answer"]],  # never empty
}
# basically in this cell we run the whole pipeline for one question, including the query reformulation step, and collect all unique contexts retrieved across the reformulated queries.
#  The final answer is generated using the original question, but the contexts are aggregated from all sub-queries. This allows us to see how the reformulation step impacts the retrieval and ultimately the answer quality.


In [14]:
# validation dataset (questions + ground truth answers)
eval_dataset = [
    {
        "question": "What is the main contribution of the transformer paper?",
        "ground_truth": "The transformer is a model architecture relying entirely on self-attention, dispensing with recurrence and convolutions entirely."
    },
    {
        "question": "What is multi-head attention?",
        "ground_truth": "Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions."
    },
    {
        "question": "What are the encoder and decoder components of the transformer?",
        "ground_truth": "The encoder maps input sequences to continuous representations. The decoder generates output sequences one element at a time."
    },
    {
        "question": "What optimizer was used to train the transformer?",
        "ground_truth": "The Adam optimizer was used with a custom learning rate schedule that increases linearly for warmup steps then decreases proportionally."
    },
    {
        "question": "What is positional encoding and why is it needed?",
        "ground_truth": "Positional encodings are added to embeddings to give the model information about token positions since the model has no recurrence or convolution."
    },
]
print(f"Eval dataset ready: {len(eval_dataset)} questions")

Eval dataset ready: 5 questions


In [16]:
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

def evaluate_pipeline(docs, chunk_size, chunk_overlap, top_k, prompt_variant):
    chain = build_rag_pipeline(docs, chunk_size, chunk_overlap, top_k, prompt_variant)

    samples = []
    for item in eval_dataset:
        out = run_rag(item["question"], chain)
        contexts = out["contexts"] if out["contexts"] else [out["answer"]]
        samples.append(SingleTurnSample(
            user_input=item["question"],
            response=out["answer"],
            retrieved_contexts=contexts,
            reference=item["ground_truth"],
        ))

    dataset = EvaluationDataset(samples=samples)

    results = evaluate(
        dataset,
        metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()]
    )

    scores_df = results.to_pandas()

    def safe_mean(col):
        if col in scores_df.columns:
            return round(float(scores_df[col].dropna().mean()), 4)
        return 0.0

    score_dict = {
        "faithfulness":      safe_mean("faithfulness"),
        "answer_relevancy":  safe_mean("answer_relevancy"),
        "context_precision": safe_mean("context_precision"),
        "context_recall":    safe_mean("context_recall"),
    }
    score_dict["avg"] = round(float(np.mean(list(score_dict.values()))), 4)

    print(f"Scores → {score_dict}")
    return score_dict

baseline_scores = evaluate_pipeline(docs, chunk_size=500, chunk_overlap=200, top_k=4, prompt_variant="v1")

C:\Users\jenil\AppData\Local\Temp\ipykernel_24496\1745997941.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\jenil\AppData\Local\Temp\ipykernel_24496\1745997941.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\jenil\AppData\Local\Temp\ipykernel_24496\1745997941.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.me

Pipeline built → chunk_size=500, top_k=4, prompt=v1, chunks=131


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[5]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[9]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[13]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[17]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')


Scores → {'faithfulness': 0.6628, 'answer_relevancy': nan, 'context_precision': 0.5036, 'context_recall': 1.0, 'avg': nan}


In [17]:
## LangGraph state defined :
class AstraState(TypedDict):
    chunk_size:    int
    chunk_overlap: int
    top_k:         int
    prompt_variant: str
    scores:        dict
    iteration:     int
    done:          bool

run_history = []  # global list to track every iteration
print("State schema defined")




State schema defined


In [18]:
## Adding three langgraph nodes for optimization loop :

SCORE_THRESHOLD = 0.85
MAX_ITERATIONS  = 2

def execute_node(state: AstraState) -> AstraState:  # Node 1: executes the pipeline and gets scores
    print(f"\n--- Iteration {state['iteration'] + 1} ---")
    print(f"Params: chunk={state['chunk_size']}, top_k={state['top_k']}, prompt={state['prompt_variant']}")
    scores = evaluate_pipeline(
        docs,
        state["chunk_size"],
        state["chunk_overlap"],
        state["top_k"],
        state["prompt_variant"]
    )
    return {**state, "scores": scores}


def evaluate_node(state: AstraState) -> AstraState: # Node 2 : evaluates scores, updates history, and checks stopping condition
    avg = state["scores"]["avg"]
    iteration = state["iteration"] + 1

    run_history.append({
        "iteration":      iteration,
        "chunk_size":     state["chunk_size"],
        "top_k":          state["top_k"],
        "prompt_variant": state["prompt_variant"],
        **state["scores"]
    })

    print(f"Avg score: {avg} | Threshold: {SCORE_THRESHOLD}")

    done = avg >= SCORE_THRESHOLD or iteration >= MAX_ITERATIONS
    return {**state, "iteration": iteration, "done": done}


def tune_node(state: AstraState) -> AstraState: # Node 3 : tunes parameters based on which specific scores are low
    scores = state["scores"]
    chunk_size    = state["chunk_size"]
    chunk_overlap = state["chunk_overlap"]
    top_k         = state["top_k"]
    prompt_variant = state["prompt_variant"]

    print("Tuning parameters...")

    # Low context recall → increase chunk size (capture more per chunk)
    if scores["context_recall"] < 0.7:
        chunk_size = int(np.clip(chunk_size + 200, 300, 1500))
        chunk_overlap = int(chunk_size * 0.2)
        print(f"  context_recall low → chunk_size bumped to {chunk_size}")

    # Low context precision → decrease top_k (fewer but cleaner chunks)
    if scores["context_precision"] < 0.7:
        top_k = int(np.clip(top_k - 1, 2, 8))
        print(f"  context_precision low → top_k reduced to {top_k}")

    # Low answer relevancy or faithfulness → switch prompt
    if scores["answer_relevancy"] < 0.7 or scores["faithfulness"] < 0.7:
        variants = ["v1", "v2", "v3"]
        current_idx = variants.index(prompt_variant)
        prompt_variant = variants[(current_idx + 1) % len(variants)]
        print(f"  relevancy/faithfulness low → prompt switched to {prompt_variant}")

    return {
        **state,
        "chunk_size":     chunk_size,
        "chunk_overlap":  chunk_overlap,
        "top_k":          top_k,
        "prompt_variant": prompt_variant,
    }

print("All three nodes defined")

All three nodes defined


In [19]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AstraState)
graph.add_node("execute",  execute_node)
graph.add_node("evaluate", evaluate_node)
graph.add_node("tune",     tune_node)

graph.set_entry_point("execute")
graph.add_edge("execute", "evaluate")

def route_after_evaluate(state: AstraState):
    if state["done"]:
        return "end"
    return "tune"

graph.add_conditional_edges(
    "evaluate",
    route_after_evaluate,
    {
        "end":  END,
        "tune": "tune",
    }
)
graph.add_edge("tune", "execute")

astra = graph.compile()

# Run Astra
final_state = astra.invoke({
    "chunk_size":     500,
    "chunk_overlap":  100,
    "top_k":          4,
    "prompt_variant": "v1",
    "scores":         {},
    "iteration":      0,
    "done":           False,
})

print("\n=== ASTRA FINISHED ===")
print(f"Final params → chunk={final_state['chunk_size']}, top_k={final_state['top_k']}, prompt={final_state['prompt_variant']}")
print(f"Final scores → {final_state['scores']}")
print(f"Iterations run → {final_state['iteration']}")


--- Iteration 1 ---
Params: chunk=500, top_k=4, prompt=v1
Pipeline built → chunk_size=500, top_k=4, prompt=v1, chunks=103


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[5]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[9]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[13]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[17]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')


IndexError: list index out of range

In [20]:
history_df = pd.DataFrame(run_history)
print(history_df.to_string(index=False))

Empty DataFrame
Columns: []
Index: []


In [21]:
# expose Astra via FastAPI
from fastapi import FastAPI
import uvicorn
import threading

api = FastAPI(title="Astra RAG Optimizer")

@api.post("/optimize")
def optimize(chunk_size: int = 500, top_k: int = 4, prompt_variant: str = "v1"):
    result = astra.invoke({
        "chunk_size":     chunk_size,
        "chunk_overlap":  int(chunk_size * 0.2),
        "top_k":          top_k,
        "prompt_variant": prompt_variant,
        "scores":         {},
        "iteration":      0,
        "done":           False,
    })
    return {
        "final_scores":     result["scores"],
        "final_params": {
            "chunk_size":     result["chunk_size"],
            "top_k":          result["top_k"],
            "prompt_variant": result["prompt_variant"],
        },
        "iterations_run":   result["iteration"],
        "full_history":     run_history,
    }

@api.get("/history")
def history():
    return {"history": run_history}

@api.get("/health")
def health():
    return {"status": "ok"}

# Run in background so notebook stays interactive
thread = threading.Thread(target=lambda: uvicorn.run(api, host="0.0.0.0", port=8000), daemon=True)
thread.start()
print("Astra API running at http://localhost:8000")
print("Docs at http://localhost:8000/docs")

Astra API running at http://localhost:8000
Docs at http://localhost:8000/docs


c:\Users\jenil\OneDrive\Desktop\Machine learning\llm_env\Lib\site-packages\uvicorn\server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-8 (<lambda>):
Traceback (most recent call last):
  File "C:\Users\jenil\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Users\jenil\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jenil\AppData\Local\Temp\ipykernel_24496\194057616.py", line 39, in <lambda>
    thread = threading.Thread(target=lambda: uvicorn.run(api, host="0.0.0.0", port=8000), daemon=True)
                                             ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jenil\OneDrive\Desktop\Machine learn